# SF-Mamba-inspired Steganalysis: Detection + Localization

A lightweight, from-scratch implementation: a residual (high-pass) front end
feeding a local CNN branch and a selective-scan (Mamba-style) global branch,
fused via a learned gate, with two heads -- cover/stego detection and
pixel-level modification localization. Includes a HILL-based embedding
simulator (generates training data + exact ground-truth localization masks
for free), an SRNet-style baseline for comparison, training, and evaluation.

**Everything in this notebook has been run cell-by-cell on this machine** on
synthetic data (no GPU / BOSSbase available here) -- see the "What's verified"
section at the end for exactly what that does and doesn't prove. Point
`COVER_DIR` at your real BOSSbase split and it should run as-is on a GPU.

Run cells top to bottom.

## 0. Setup

In [1]:
!pip install -q torch numpy scipy Pillow scikit-learn tqdm 2>/dev/null
import torch, numpy as np
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

torch: 2.4.1+cu121 | CUDA available: False
Using device: cpu


## 1. Residual front end (SRM-style high-pass filters)

Suppresses image content, exposes the embedding residual -- every serious
steganalysis CNN (Xu-Net, SRNet, Zhu-Net) starts this way. Filters are fixed
(not trained) by default; see the code comment for why.

In [2]:
"""
Residual front end: fixed high-pass filter bank (SRM-style) + learnable 1x1 projection.

Why this exists (design rationale):
Steganalysis has to suppress natural image *content* and expose the much weaker
embedding *residual*. Every serious steganalysis CNN (Xu-Net, SRNet, Zhu-Net) starts
this way. We use a small, well-known set of high-pass kernels rather than the full
30-kernel SRM bank to keep parameter count and compute down -- you can expand
SRM_KERNELS below if you want the full bank later.

IMPORTANT: the high-pass kernels are FIXED (not trained) by default. This is
intentional -- letting gradient descent "relax" the high-pass constraint early in
training is a known way for the network to fall back on content features instead of
residual features. You can unfreeze them later (see `learnable=True`) once the rest
of the network has converged, if you want to experiment with that.
"""
import torch
import torch.nn as nn
import numpy as np


def _build_srm_kernels():
    """A compact bank of well-known high-pass kernels used across the steganalysis
    literature (subset of the classic 30-kernel SRM bank, kept small on purpose).
    Each kernel is normalized so filter response magnitude is comparable.
    """
    kernels = []

    # 1st-order and 2nd-order pixel difference kernels (KV / SRM staples)
    k1 = np.array([[0, 0, 0], [0, -1, 1], [0, 0, 0]], dtype=np.float32)
    k2 = np.array([[0, 0, 0], [0, -1, 0], [0, 1, 0]], dtype=np.float32)
    k3 = np.array([[0, 0, 0], [0, 1, -2], [0, 0, 1]], dtype=np.float32)  # placeholder-style 3rd order
    k4 = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], dtype=np.float32) / 4.0  # KB / HILL kernel
    k5 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32) / 4.0      # Laplacian
    k6 = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=np.float32) / 4.0    # Sobel-y
    k7 = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=np.float32) / 4.0    # Sobel-x

    base = [k1, k2, k3, k4, k5, k6, k7]

    # Add 4 rotations of a couple of the directional ones for richer coverage
    for k in [k1, k2]:
        for _ in range(3):
            k = np.rot90(k).copy()
            base.append(k.astype(np.float32))

    kernels = np.stack(base, axis=0)  # (N, 3, 3)
    return kernels


SRM_KERNELS = _build_srm_kernels()  # (N, 3, 3)


class ResidualStem(nn.Module):
    def __init__(self, out_channels: int = 32, learnable: bool = False):
        super().__init__()
        n_kernels = SRM_KERNELS.shape[0]
        weight = torch.from_numpy(SRM_KERNELS).unsqueeze(1)  # (N, 1, 3, 3)

        self.hp_conv = nn.Conv2d(1, n_kernels, kernel_size=3, padding=1, bias=False)
        with torch.no_grad():
            self.hp_conv.weight.copy_(weight)
        self.hp_conv.weight.requires_grad = learnable

        # TLU-style clipping is common in SRNet/Zhu-Net to bound the residual range
        # before the network amplifies outliers.
        self.clip_value = 3.0

        self.project = nn.Sequential(
            nn.Conv2d(n_kernels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # x: (B, 1, H, W) in roughly [-1, 1] or [0, 1] -- see dataset normalization
        r = self.hp_conv(x)
        r = torch.clamp(r, -self.clip_value, self.clip_value)
        return self.project(r)  # (B, out_channels, H, W)


# quick check
_m = ResidualStem(out_channels=32)
_x = torch.randn(2, 1, 256, 256)
print("ResidualStem output:", _m(_x).shape)


ResidualStem output: torch.Size([2, 32, 256, 256])


## 2. Local CNN branch

Small residual-block stack operating directly on the residual features -- deliberately lightweight (SRNet-scale, not ResNet-50-scale) given how small steganalysis datasets are.

In [3]:
"""
Local CNN branch: a handful of residual blocks operating directly on the
high-pass residual features. This is deliberately small (SRNet-style, not
ResNet-50-style) -- steganalysis datasets are tiny (10k images) so a heavy
ImageNet-scale backbone will overfit, and it also carries semantic-content
inductive bias you don't want here (see the chat discussion on why the
original SF-Mamba ResNet-50 backbone is a bad fit).
"""
import torch
import torch.nn as nn


class ResBlock(nn.Module):
    def __init__(self, channels: int, stride: int = 1):
        super().__init__()
        out_channels = channels * stride if stride > 1 else channels
        self.conv1 = nn.Conv2d(channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = None
        if stride > 1 or channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class LocalBranch(nn.Module):
    """
    4 residual blocks, one downsampling step (stride 2 once) to keep the
    feature map large enough (>=64x64 for a 256x256 input) for the
    localization head to have enough spatial resolution to work with.
    """
    def __init__(self, in_channels: int = 32, base_channels: int = 64, n_blocks: int = 4):
        super().__init__()
        blocks = [ResBlock(in_channels, stride=1)]
        # project in_channels -> base_channels on block 2 with a stride-2 downsample
        blocks.append(ResBlock(in_channels, stride=2))  # in_channels*2 out
        c = in_channels * 2
        for _ in range(n_blocks - 2):
            blocks.append(ResBlock(c, stride=1))
        self.blocks = nn.Sequential(*blocks)
        self.out_channels = c

    def forward(self, x):
        return self.blocks(x)


# quick check
_stem = ResidualStem(out_channels=32)
_branch = LocalBranch(in_channels=32)
_x = torch.randn(2, 1, 256, 256)
_f = _branch(_stem(_x))
print("LocalBranch output:", _f.shape, "channels:", _branch.out_channels)


LocalBranch output: torch.Size([2, 64, 128, 128]) channels: 64


## 3. Global selective-scan branch (simplified Mamba/VMamba-style VSS block)

Pure-PyTorch reference implementation (no `mamba-ssm` CUDA dependency, so it
runs anywhere). Correct but slow -- see the note in the code about swapping in
the real CUDA kernel once you have a GPU and are training at full scale.

In [ ]:
"""
A simplified, pure-PyTorch 2D selective-scan block (S6 core), in the spirit of
Mamba / VMamba's VSS block, but deliberately scaled down for a steganalysis
setting (small dataset, small images, no CUDA selective-scan kernel dependency).

WHY THIS IS SIMPLIFIED, ON PURPOSE:
- The official Mamba/VMamba selective-scan is a fused CUDA kernel (mamba-ssm
  package). We don't depend on it here so this runs anywhere PyTorch runs
  (including CPU, for debugging). If you later want real training-time speed,
  swap `_selective_scan_sequential` below for `mamba_ssm.ops.selective_scan_fn`.
- A python for-loop over the sequence length is used for the recurrence. This
  is CORRECT but SLOW for long sequences. To keep it tractable without a CUDA
  kernel, we downsample the feature map before scanning (see `pre_downsample`)
  and upsample the output back afterward. This mirrors how VMamba itself
  applies VSS blocks at progressively downsampled stages -- we're just being
  more aggressive about it for compute-budget reasons.

DESIGN (per the spec you pasted):
- Cross-scan: unfold the 2D feature map into multiple 1D sequences (here: 2
  directions -- row-major forward and its reverse -- rather than the full
  4-direction VMamba cross-scan, to keep params/compute down as specified).
- S6 block: input-dependent Delta/B/C, fixed per-channel A, diagonal state.
- Cross-merge: fold the two directional outputs back to 2D and average them.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def _selective_scan_sequential(x, delta, A, B, C, D):
    """
    Reference (slow, pure-PyTorch) selective scan.

    x:     (B, C, L)   input sequence, per channel
    delta: (B, C, L)   input-dependent step size (already softplus'd, >0)
    A:     (C, N)      fixed per-channel state matrix (negative, diagonal)
    B:     (B, N, L)   input-dependent, shared across channels
    C:     (B, N, L)   input-dependent, shared across channels
    D:     (C,)        skip-connection weight per channel

    returns y: (B, C, L)
    """
    Bs, Ch, L = x.shape
    N = A.shape[1]
    device, dtype = x.device, x.dtype

    # discretize: a_t = exp(delta_t * A)   -> (B, C, L, N)
    # done lazily per-timestep inside the loop to keep memory bounded
    h = torch.zeros(Bs, Ch, N, device=device, dtype=dtype)
    ys = []
    for t in range(L):
        delta_t = delta[:, :, t].unsqueeze(-1)          # (B, C, 1)
        a_t = torch.exp(delta_t * A.unsqueeze(0))        # (B, C, N)
        Bx_t = delta_t * B[:, :, t].unsqueeze(1) * x[:, :, t].unsqueeze(-1)  # (B, C, N)
        h = a_t * h + Bx_t                                # (B, C, N)
        y_t = (h * C[:, :, t].unsqueeze(1)).sum(-1)        # (B, C)
        ys.append(y_t)
    y = torch.stack(ys, dim=-1)  # (B, C, L)
    y = y + x * D.view(1, -1, 1)
    return y


class S6Core(nn.Module):
    def __init__(self, channels: int, state_dim: int = 8, dt_rank: int = 8):
        super().__init__()
        self.channels = channels
        self.state_dim = state_dim

        # A is fixed, learnable, constrained negative via -softplus(param)
        self.A_log = nn.Parameter(torch.randn(channels, state_dim) * 0.1)
        self.D = nn.Parameter(torch.ones(channels))

        # input-dependent projections for delta, B, C
        self.x_proj = nn.Linear(channels, dt_rank + 2 * state_dim, bias=False)
        self.dt_proj = nn.Linear(dt_rank, channels, bias=True)

    def forward(self, x_seq):
        # x_seq: (B, L, C)
        Bsz, L, Ch = x_seq.shape
        proj = self.x_proj(x_seq)  # (B, L, dt_rank + 2N)
        dt_rank = proj.shape[-1] - 2 * self.state_dim
        delta_in, Bc, Cc = torch.split(proj, [dt_rank, self.state_dim, self.state_dim], dim=-1)

        delta = F.softplus(self.dt_proj(delta_in))          # (B, L, C)
        A = -F.softplus(self.A_log)                         # (C, N), strictly negative

        x_t = x_seq.transpose(1, 2)          # (B, C, L)
        delta_t = delta.transpose(1, 2)      # (B, C, L)
        B_t = Bc.transpose(1, 2)             # (B, N, L)
        C_t = Cc.transpose(1, 2)             # (B, N, L)

        y = _selective_scan_sequential(x_t, delta_t, A, B_t, C_t, self.D)  # (B, C, L)
        return y.transpose(1, 2)  # (B, L, C)


class VSSBlock2D(nn.Module):
    """
    One 2D selective-scan block: LayerNorm -> linear expand -> cross-scan (2
    directions) -> S6 -> cross-merge -> depthwise conv -> linear project ->
    residual add. Operates at a downsampled resolution for tractability, then
    upsamples back to `target_hw`.
    """
    def __init__(self, in_channels: int, hidden_channels: int = 64,
                 state_dim: int = 8, pre_downsample: int = 4):
        super().__init__()
        self.pre_downsample = pre_downsample
        self.norm_in = nn.GroupNorm(1, in_channels)
        self.down = nn.Conv2d(in_channels, hidden_channels, kernel_size=pre_downsample,
                               stride=pre_downsample)
        self.s6 = S6Core(hidden_channels, state_dim=state_dim)
        self.dwconv = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3,
                                 padding=1, groups=hidden_channels)
        self.act = nn.SiLU()
        self.out_proj = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1)

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        x = self.norm_in(x)
        z = self.down(x)  # (B, hidden, H/k, W/k)
        _, Ch, h, w = z.shape

        # --- cross-scan: 2 directions (row-major fwd, row-major reverse) ---
        seq_fwd = z.flatten(2).transpose(1, 2)          # (B, L, Ch)
        seq_rev = torch.flip(seq_fwd, dims=[1])          # reversed order

        y_fwd = self.s6(seq_fwd)                          # (B, L, Ch)
        y_rev = torch.flip(self.s6(seq_rev), dims=[1])    # un-reverse to align

        # --- cross-merge: average the two directions ---
        y = (y_fwd + y_rev) * 0.5
        y = y.transpose(1, 2).reshape(B, Ch, h, w)

        y = self.act(self.dwconv(y))
        y = self.out_proj(y)

        # upsample back to input resolution
        y = F.interpolate(y, size=(H, W), mode="bilinear", align_corners=False)
        return y


# quick check
import time as _time
_m = VSSBlock2D(in_channels=32, hidden_channels=64, pre_downsample=8)
_x = torch.randn(2, 32, 256, 256)
_t0 = _time.time()
_y = _m(_x)
print("VSSBlock2D output:", _y.shape, "took", round(_time.time() - _t0, 2), "s")


VSSBlock2D output: torch.Size([2, 64, 256, 256]) took 1.18 s


## 4. Residual-aware gated fusion

Replaces SF-Mamba's SFAM/GFAM (which are built around semantic foreground/background decoupling -- no analog in steganalysis) with a simpler gate that learns how much to trust local vs. global evidence per location.

In [5]:
"""
Residual-aware gated fusion (your spec's replacement for SFAM/GFAM).

Rationale: SFAM/GFAM in the original SF-Mamba are built around semantic
foreground/background decoupling (buildings vs. roads vs. clutter), which has
no analog in steganalysis. This gate instead learns, per spatial location, how
much to trust local residual-CNN evidence vs. globally-aggregated evidence --
a much simpler and more directly justifiable mechanism for this task.
"""
import torch
import torch.nn as nn


class GatedFusion(nn.Module):
    def __init__(self, local_channels: int, global_channels: int, out_channels: int):
        super().__init__()
        self.local_proj = nn.Conv2d(local_channels, out_channels, 1, bias=False)
        self.global_proj = nn.Conv2d(global_channels, out_channels, 1, bias=False)
        self.gate = nn.Sequential(
            nn.Conv2d(local_channels + global_channels, out_channels, 1),
            nn.Sigmoid(),
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU(inplace=True)

    def forward(self, f_local, f_global):
        cat = torch.cat([f_local, f_global], dim=1)
        g = self.gate(cat)  # (B, out_channels, H, W) in (0,1)
        fused = g * self.local_proj(f_local) + (1 - g) * self.global_proj(f_global)
        return self.act(self.bn(fused))


# quick check
_m = GatedFusion(local_channels=64, global_channels=64, out_channels=96)
_fl = torch.randn(2, 64, 128, 128)
_fg = torch.randn(2, 64, 128, 128)
print("GatedFusion output:", _m(_fl, _fg).shape)


GatedFusion output: torch.Size([2, 96, 128, 128])


## 5. Detection + localization heads

In [ ]:
import torch
import torch.nn as nn


class DetectionHead(nn.Module):
    """Global average pool -> FC -> single logit (cover=0 / stego=1)."""
    def __init__(self, in_channels: int):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(in_channels, 1)

    def forward(self, x):
        z = self.gap(x).flatten(1)
        return self.fc(z).squeeze(-1)  # (B,) raw logits -- use BCEWithLogitsLoss


class LocalizationHead(nn.Module):
    """Small decoder: upsample back to input resolution, 1x1 conv to 1 channel.
    Outputs raw logits (B, 1, H, W) -- use BCEWithLogitsLoss / Dice on sigmoid(out).
    """
    def __init__(self, in_channels: int, upsample_factor: int, mid_channels: int = 32):
        super().__init__()
        n_steps = max(1, (upsample_factor - 1).bit_length())  # e.g. factor=2 -> 1 step
        layers = []
        c = in_channels
        for _ in range(n_steps):
            layers += [
                nn.Conv2d(c, mid_channels, 3, padding=1),
                nn.BatchNorm2d(mid_channels),
                nn.ReLU(inplace=True),
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ]
            c = mid_channels
        layers.append(nn.Conv2d(c, 1, kernel_size=1))
        self.decoder = nn.Sequential(*layers)
        self.upsample_factor = upsample_factor

    def forward(self, x, target_hw):
        out = self.decoder(x)
        if out.shape[-2:] != target_hw:
            out = nn.functional.interpolate(out, size=target_hw, mode="bilinear",
                                             align_corners=False)
        return out  # (B, 1, H, W) logits


# quick check
_fused = torch.randn(2, 96, 128, 128)
_det = DetectionHead(96)
_loc = LocalizationHead(96, upsample_factor=2)
print("Detection logits:", _det(_fused).shape)
print("Localization logits:", _loc(_fused, target_hw=(256, 256)).shape)


Detection logits: torch.Size([2])
Localization logits: torch.Size([2, 1, 256, 256])


## 6. Full model assembly

In [7]:
import torch
import torch.nn as nn

class SFMambaSteganalysis(nn.Module):
    """
    Full pipeline:
      input -> ResidualStem -> {LocalBranch, VSSBlock2D} -> GatedFusion
             -> DetectionHead
             -> LocalizationHead
    """
    def __init__(self,
                 stem_channels: int = 32,
                 local_channels: int = 64,
                 global_hidden: int = 64,
                 fused_channels: int = 96,
                 vss_pre_downsample: int = 8,
                 n_local_blocks: int = 4):
        super().__init__()
        self.stem = ResidualStem(out_channels=stem_channels)
        self.local_branch = LocalBranch(in_channels=stem_channels,
                                         n_blocks=n_local_blocks)
        # global branch runs on the stem output directly (not the downsampled
        # local-branch output) so it sees full-resolution residual evidence,
        # then internally downsamples for scan tractability (see vss_block.py)
        self.global_branch = VSSBlock2D(in_channels=stem_channels,
                                         hidden_channels=global_hidden,
                                         pre_downsample=vss_pre_downsample)

        # local branch downsamples the stem output by 2x once; make the
        # global branch output match that resolution before fusion
        self.local_downsample_factor = 2

        self.fusion = GatedFusion(local_channels=self.local_branch.out_channels,
                                   global_channels=global_hidden,
                                   out_channels=fused_channels)

        self.detection_head = DetectionHead(fused_channels)
        self.localization_head = LocalizationHead(fused_channels,
                                                    upsample_factor=self.local_downsample_factor)

    def forward(self, x):
        # x: (B, 1, H, W)
        H, W = x.shape[-2:]
        stem_out = self.stem(x)                      # (B, stem_c, H, W)

        f_local = self.local_branch(stem_out)         # (B, local_c, H/2, W/2)
        f_global_full = self.global_branch(stem_out)  # (B, global_hidden, H, W)
        f_global = nn.functional.interpolate(
            f_global_full, size=f_local.shape[-2:], mode="bilinear", align_corners=False
        )                                              # match local branch resolution

        fused = self.fusion(f_local, f_global)         # (B, fused_c, H/2, W/2)

        det_logits = self.detection_head(fused)                       # (B,)
        loc_logits = self.localization_head(fused, target_hw=(H, W))  # (B, 1, H, W)
        return det_logits, loc_logits


# quick check
_model = SFMambaSteganalysis()
_n_params = sum(p.numel() for p in _model.parameters())
print(f"Total params: {_n_params:,} ({_n_params/1e6:.2f}M)")
_x = torch.randn(2, 1, 256, 256)
_det, _loc = _model(_x)
print("Detection logits:", _det.shape)
print("Localization logits:", _loc.shape)


Total params: 416,279 (0.42M)


Detection logits: torch.Size([2])
Localization logits: torch.Size([2, 1, 256, 256])


### Sanity check: forward + backward pass, gradient flow

In [8]:
model = SFMambaSteganalysis().to(DEVICE)
x = torch.randn(4, 1, 256, 256, device=DEVICE)
y_det = torch.randint(0, 2, (4,), device=DEVICE).float()
y_loc = torch.randint(0, 2, (4, 1, 256, 256), device=DEVICE).float()

det_logits, loc_logits = model(x)
loss = (torch.nn.functional.binary_cross_entropy_with_logits(det_logits, y_det)
        + torch.nn.functional.binary_cross_entropy_with_logits(loc_logits, y_loc))
loss.backward()

print("loss:", loss.item())
print("high-pass stem frozen (should be False, None grad):",
      model.stem.hp_conv.weight.requires_grad, model.stem.hp_conv.weight.grad)
print("local branch gradient flowing (should be > 0):",
      model.local_branch.blocks[0].conv1.weight.grad.abs().mean().item())

loss: 1.3979440927505493
high-pass stem frozen (should be False, None grad): False None
local branch gradient flowing (should be > 0): 0.0005524645675905049


## 7. HILL content-adaptive cost function

Cost is inversely proportional to local texture complexity -- smooth regions
get high cost (avoid), textured regions get low cost (embed here, changes
blend into existing noise). Reference: Li, Wang, Huang, Shi, ICIP 2014.

In [9]:
"""
HILL (HIgh-pass, Low-pass, Low-pass) cost function.
Reference: Li, Wang, Huang, Shi, "A New Cost Function for Spatial Image
Steganography," ICIP 2014.

Cost at each pixel is INVERSELY proportional to local texture complexity:
smooth/flat regions -> high cost (avoid embedding there, changes are visible
as statistical anomalies), textured regions -> low cost (embed here, changes
blend into existing noise).

Pipeline:
  1. High-pass filter the image with a small kernel (H1) to get a residual.
  2. Take the absolute value.
  3. Smooth twice with average filters of increasing size (L1 then L2) --
     this is what turns a single-pixel residual into a *neighborhood*
     texture-complexity estimate.
  4. Cost = 1 / (smoothed result), with a floor to avoid divide-by-zero.

This is the simplest of the standard content-adaptive cost functions
(WOW and S-UNIWARD use directional filter banks and are more involved --
same interface, swap this module out if/when you add them).
"""
import numpy as np
from scipy.signal import convolve2d


# The HILL paper's H1 kernel (a normalized Laplacian-of-Gaussian-like kernel)
H1 = np.array([[-1, 2, -1],
               [2, -4, 2],
               [-1, 2, -1]], dtype=np.float64)

L1 = np.ones((3, 3), dtype=np.float64) / 9.0
L2 = np.ones((15, 15), dtype=np.float64) / 225.0

_EPS = 1e-10


def compute_hill_cost(image: np.ndarray) -> np.ndarray:
    """
    image: 2D float array, grayscale, any consistent range (e.g. 0-255).
    returns: cost map, same shape, cost(i,j) > 0 for every pixel.
    """
    assert image.ndim == 2, "expects a single-channel (grayscale) image"
    img = image.astype(np.float64)

    residual = convolve2d(img, H1, mode="same", boundary="symm")
    abs_residual = np.abs(residual)

    smoothed = convolve2d(abs_residual, L1, mode="same", boundary="symm")
    smoothed = convolve2d(smoothed, L2, mode="same", boundary="symm")

    cost = 1.0 / (smoothed + _EPS)

    # Boundary pixels where the embedding simulator forbids saturated values
    # (0 or 255) get an effectively infinite cost so they're never selected.
    cost[img <= 0] = 1e13
    cost[img >= 255] = 1e13
    return cost


# quick check
_rng = np.random.default_rng(0)
_fake_img = _rng.integers(0, 256, size=(64, 64)).astype(np.float64)
_cost = compute_hill_cost(_fake_img)
print("cost map shape:", _cost.shape, "min/max:", _cost.min(), _cost.max())


cost map shape: (64, 64) min/max: 0.002300712823320631 10000000000000.0


## 8. Ternary embedding simulator

Standard steganalysis-research methodology (used by SRNet, Zhu-Net, and the
original WOW/S-UNIWARD/HILL papers themselves): simulates the rate-distortion
trade-off of a near-optimal STC encoder without needing an actual STC
implementation. Gives you the stego image, the exact hard modification mask
(free ground truth for localization), and a soft per-pixel probability map.

In [10]:
"""
Ternary (+-1) embedding simulator.

This is the standard way steganalysis research generates training data: rather
than running a real STC (syndrome-trellis code) encoder -- which requires an
actual message payload and is computationally heavier -- everyone in the field
(SRNet, Zhu-Net, and the original WOW/S-UNIWARD/HILL papers themselves) uses a
*simulator* that achieves the same rate-distortion trade-off that a real
near-optimal STC encoder would, without needing an actual STC implementation.
This is the correct, standard, published methodology -- not an approximation
you're introducing yourself.

How it works:
  Given a per-pixel cost rho(i,j) (from HILL, WOW, etc.) and a target payload
  alpha (bits per pixel), find a scalar "inverse temperature" lambda such that
  the expected total embedding capacity (in bits) at that lambda equals the
  target payload. Then, independently at each pixel, sample a ternary change
  in {-1, 0, +1} with probability (p, 1-2p, p) where
      p(i,j) = exp(-lambda * rho(i,j)) / (1 + 2*exp(-lambda * rho(i,j)))
  Higher-cost pixels get near-zero p (never touched); lower-cost pixels get p
  approaching 1/3 (heavily used) as payload increases.

This gives you, for free:
  - the stego image (cover + sampled change)
  - the EXACT hard modification mask (change != 0) -- ground truth localization
  - the soft per-pixel probability map, if you want soft localization targets
"""
import numpy as np


def _capacity_bits(rho: np.ndarray, lam: float) -> float:
    """Total expected embedding capacity (bits) at a given lambda."""
    p = np.exp(-lam * rho) / (1.0 + 2.0 * np.exp(-lam * rho))
    p = np.clip(p, 1e-12, (1.0 / 3.0) - 1e-12)  # keep entropy well-defined
    p0 = 1.0 - 2.0 * p
    # ternary entropy per pixel, in bits
    h = -2 * p * np.log2(p) - p0 * np.log2(p0)
    return float(h.sum())


def _solve_lambda(rho: np.ndarray, target_bits: float,
                   lam_lo: float = 1e-6, lam_hi: float = 500.0,
                   n_iters: int = 40) -> float:
    """Binary search for lambda such that capacity(lambda) == target_bits.
    Capacity is monotonically decreasing in lambda (higher lambda -> more
    conservative embedding -> lower capacity), so binary search is valid.
    """
    # make sure bracket is valid; expand hi if needed for very low payloads
    cap_lo = _capacity_bits(rho, lam_lo)
    if target_bits >= cap_lo:
        return lam_lo  # target payload exceeds max achievable capacity at lam_lo

    cap_hi = _capacity_bits(rho, lam_hi)
    tries = 0
    while cap_hi > target_bits and tries < 20:
        lam_hi *= 2
        cap_hi = _capacity_bits(rho, lam_hi)
        tries += 1

    for _ in range(n_iters):
        lam_mid = 0.5 * (lam_lo + lam_hi)
        cap_mid = _capacity_bits(rho, lam_mid)
        if cap_mid > target_bits:
            lam_lo = lam_mid
        else:
            lam_hi = lam_mid
    return 0.5 * (lam_lo + lam_hi)


def simulate_embedding(cover: np.ndarray, rho: np.ndarray, payload_bpp: float,
                        rng: np.random.Generator = None):
    """
    cover: 2D uint8/float array, grayscale cover image
    rho:   2D float array, same shape, cost map (e.g. from compute_hill_cost)
    payload_bpp: target payload in bits per pixel
    rng: numpy random Generator (pass one in for reproducibility)

    Returns:
      stego:      2D array, same dtype range as cover, modified image
      hard_mask:  2D uint8 array, 1 where pixel was changed, 0 otherwise
      change:     2D int array in {-1, 0, +1}, the actual signed change applied
      p_map:      2D float array, the per-pixel total change probability (2p),
                  useful as a SOFT localization target
    """
    if rng is None:
        rng = np.random.default_rng()

    H, W = cover.shape
    target_bits = payload_bpp * H * W
    lam = _solve_lambda(rho, target_bits)

    p = np.exp(-lam * rho) / (1.0 + 2.0 * np.exp(-lam * rho))
    p = np.clip(p, 0.0, 1.0 / 3.0)

    # sample change in {-1, 0, +1} per pixel with prob (p, 1-2p, p)
    u = rng.random(size=cover.shape)
    change = np.zeros_like(cover, dtype=np.int8)
    change[u < p] = -1
    change[(u >= p) & (u < 2 * p)] = +1
    # else: stays 0

    stego = cover.astype(np.int16) + change
    stego = np.clip(stego, 0, 255).astype(cover.dtype)

    hard_mask = (change != 0).astype(np.uint8)
    p_map = (2 * p).astype(np.float32)  # total probability of being touched

    return stego, hard_mask, change, p_map


if __name__ == "__main__":
    
    rng = np.random.default_rng(0)
    cover = rng.integers(0, 256, size=(64, 64)).astype(np.uint8)
    rho = compute_hill_cost(cover)

    for bpp in [0.1, 0.2, 0.4]:
        stego, mask, change, p_map = simulate_embedding(cover, rho, bpp, rng)
        actual_rate = mask.sum() / cover.size
        print(f"target bpp={bpp:.2f} -> fraction of pixels changed={actual_rate:.4f}, "
              f"max |change|={np.abs(change).max()}")


target bpp=0.10 -> fraction of pixels changed=0.0090, max |change|=1
target bpp=0.20 -> fraction of pixels changed=0.0227, max |change|=1
target bpp=0.40 -> fraction of pixels changed=0.0698, max |change|=1


### Verify content-adaptive behavior

Confirms embedding actually concentrates in textured regions, not uniformly
-- half the test image is a smooth gradient, half is noise/texture.

In [11]:
_x_grad = np.linspace(0, 255, 64)
_img = np.tile(_x_grad, (64, 1))
_noise_patch = _rng.integers(0, 256, size=(64, 64)).astype(np.float64)
_img[:, :32] = _noise_patch[:, :32]  # left half textured, right half smooth
_img = _img.astype(np.uint8)

_rho2 = compute_hill_cost(_img)
print("mean cost, textured half:", _rho2[:, :32].mean())
print("mean cost, smooth half:  ", _rho2[:, 32:].mean())

for _bpp in [0.1, 0.2, 0.4]:
    _stego, _mask, _change, _p_map = simulate_embedding(_img, _rho2, _bpp, _rng)
    _rate = _mask.sum() / _img.size
    _frac_textured = _mask[:, :32].sum() / _mask.sum() if _mask.sum() > 0 else 0
    print(f"bpp={_bpp}: change_rate={_rate:.4f}, frac of changes in textured half={_frac_textured:.3f}")

mean cost, textured half: 107421875000.00313
mean cost, smooth half:   319375000000.01636
bpp=0.1: change_rate=0.0117, frac of changes in textured half=1.000
bpp=0.2: change_rate=0.0339, frac of changes in textured half=1.000
bpp=0.4: change_rate=0.0811, frac of changes in textured half=0.985


## 9. On-the-fly cover/stego PyTorch Dataset

Generates a fresh stego realization at `__getitem__` time (free augmentation,
easy payload sweeps, matches standard practice). Point `cover_dir` at a
directory of grayscale images (BOSSbase/BOWS2 `.pgm`/`.png`).

In [12]:
"""
On-the-fly cover/stego dataset.

Design choice: generate stego images ON THE FLY at __getitem__ time rather than
pre-generating a fixed stego dataset on disk. Reasons:
  1. Every epoch sees a fresh random embedding realization of the same cover
     image (different random change locations for the same p-map), which acts
     as a free form of data augmentation specific to this task.
  2. You can trivially sweep payload rates / algorithms without regenerating
     and storing multiple copies of the dataset on disk (BOSSbase alone at
     4 payload levels would otherwise mean 5x the storage: 1 cover + 4 stego
     sets).
  3. Matches how SRNet/Zhu-Net and most steganalysis papers actually do it.

Trade-off: this means a fresh HILL cost computation per image per epoch. HILL
is cheap (a few small convolutions), but if you switch to a more expensive
cost function later, consider caching cost maps to disk keyed by image id.

Expected directory layout:
    root/
      cover/
        0001.pgm (or .png)
        0002.pgm
        ...

Payload rate is either fixed or sampled per-item from a provided list, which
lets a single training run cover multiple payloads (helps generalization,
matches "train across payloads" setups common in the literature).
"""
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

# (compute_hill_cost / simulate_embedding already defined above in this notebook)


class SteganalysisDataset(Dataset):
    def __init__(self, cover_dir: str, payload_bpps=(0.1, 0.2, 0.3, 0.4),
                 image_size: int = 256, seed: int = None, extensions=(".pgm", ".png", ".pnm")):
        self.paths = []
        for ext in extensions:
            self.paths.extend(sorted(glob.glob(os.path.join(cover_dir, f"*{ext}"))))
        if not self.paths:
            raise FileNotFoundError(
                f"No cover images found under {cover_dir} with extensions {extensions}. "
                f"Point cover_dir at your BOSSbase/BOWS2 folder."
            )
        self.payload_bpps = list(payload_bpps)
        self.image_size = image_size
        # Each worker process gets its own generator (seeded per-worker in
        # DataLoader's worker_init_fn -- see get_dataloader() below) so
        # random embedding realizations aren't duplicated across workers.
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.paths)

    def set_rng(self, rng: np.random.Generator):
        self.rng = rng

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")
        if img.size != (self.image_size, self.image_size):
            img = img.resize((self.image_size, self.image_size), Image.BICUBIC)
        cover = np.array(img, dtype=np.uint8)

        # randomly decide cover (label 0) vs stego (label 1) for this sample --
        # 50/50 is standard for steganalysis training
        is_stego = self.rng.random() < 0.5

        if is_stego:
            bpp = self.rng.choice(self.payload_bpps)
            rho = compute_hill_cost(cover)
            image, hard_mask, _, soft_mask = simulate_embedding(cover, rho, bpp, self.rng)
            y_det = 1.0
        else:
            image = cover
            hard_mask = np.zeros_like(cover, dtype=np.uint8)
            soft_mask = np.zeros_like(cover, dtype=np.float32)
            y_det = 0.0

        # normalize to roughly [-1, 1] for the network's residual stem
        image_t = torch.from_numpy(image.astype(np.float32) / 127.5 - 1.0).unsqueeze(0)
        hard_mask_t = torch.from_numpy(hard_mask.astype(np.float32)).unsqueeze(0)
        soft_mask_t = torch.from_numpy(soft_mask.astype(np.float32)).unsqueeze(0)
        y_det_t = torch.tensor(y_det, dtype=torch.float32)

        return {
            "image": image_t,
            "y_det": y_det_t,
            "hard_mask": hard_mask_t,
            "soft_mask": soft_mask_t,
            "path": self.paths[idx],
        }


def _worker_init_fn(worker_id):
    """Give each DataLoader worker an independent, reproducible RNG stream."""
    worker_info = torch.utils.data.get_worker_info()
    base_seed = torch.initial_seed() % (2**32)
    worker_info.dataset.set_rng(np.random.default_rng(base_seed + worker_id))


def get_dataloader(cover_dir, payload_bpps=(0.1, 0.2, 0.3, 0.4), image_size=256,
                    batch_size=16, num_workers=4, shuffle=True, seed=None):
    ds = SteganalysisDataset(cover_dir, payload_bpps=payload_bpps,
                              image_size=image_size, seed=seed)
    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
        worker_init_fn=_worker_init_fn if num_workers > 0 else None,
        pin_memory=True, drop_last=shuffle,
    )


# usage examples further down in this notebook


### Make synthetic cover images for a quick smoke test

Replace `COVER_DIR` with a real BOSSbase/BOWS2 folder for actual training.

In [13]:
import os
from PIL import Image

COVER_DIR = "/tmp/fake_cover"   # <-- change to your real cover-image directory
os.makedirs(COVER_DIR, exist_ok=True)

_rng_data = np.random.default_rng(1)
for i in range(6):
    img = _rng_data.integers(0, 256, size=(256, 256)).astype(np.uint8)
    Image.fromarray(img).save(f"{COVER_DIR}/{i:04d}.png")
print("wrote 6 synthetic cover images to", COVER_DIR)

wrote 6 synthetic cover images to /tmp/fake_cover


In [14]:
loader = get_dataloader(COVER_DIR, batch_size=4, num_workers=0, payload_bpps=(0.4,))
batch = next(iter(loader))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype, "range:", v.min().item(), v.max().item())
print("y_det:", batch["y_det"])
print("mean hard_mask per sample:", batch["hard_mask"].mean(dim=(1, 2, 3)))

image (4, 1, 256, 256) torch.float32 range: -1.0 1.0
y_det (4,) torch.float32 range: 0.0 1.0
hard_mask (4, 1, 256, 256) torch.float32 range: 0.0 1.0
soft_mask (4, 1, 256, 256) torch.float32 range: 0.0 0.1313716024160385
y_det: tensor([0., 0., 1., 0.])
mean hard_mask per sample: tensor([0.0000, 0.0000, 0.0626, 0.0000])


## 10. Loss functions

`L_total = L_det + alpha * L_loc`, where `L_loc` combines weighted BCE and
Dice -- necessary because at realistic payloads (0.1-0.4 bpp) only a few
percent of pixels are ever modified, so plain BCE collapses to predicting
"nothing changed" everywhere.

In [15]:
"""
Loss functions for the joint detection + localization model.

L_total = L_det + alpha * L_loc
L_loc   = lambda_bce * WeightedBCE(pred, soft_target) + lambda_dice * Dice(pred, hard_target)

Why weighted BCE + Dice, not plain BCE:
At realistic payloads (0.1-0.4 bpp), only a few percent of pixels are ever
modified (see the simulator sanity check -- ~1-8% of pixels for 0.1-0.4 bpp).
Plain BCE on a >90%-negative pixel-classification problem collapses to
predicting "unchanged everywhere" and still gets a very low loss. Weighting
the positive class and adding a Dice term (which is scale-invariant to class
imbalance, unlike BCE) are both standard fixes for this in segmentation
literature, and directly necessary here.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def weighted_bce_loss(logits: torch.Tensor, soft_target: torch.Tensor,
                       pos_weight: float = None) -> torch.Tensor:
    """
    logits: (B, 1, H, W) raw localization logits
    soft_target: (B, 1, H, W) in [0, 1] -- can be the exact hard mask or the
        p_map soft probability map from the simulator
    pos_weight: if None, computed per-batch from the target's positive rate
        (so you don't have to hand-tune it per payload level)
    """
    if pos_weight is None:
        pos_rate = soft_target.mean().clamp(min=1e-4, max=1 - 1e-4)
        pos_weight = ((1 - pos_rate) / pos_rate).detach()
    pos_weight_t = torch.as_tensor(pos_weight, device=logits.device, dtype=logits.dtype)
    return F.binary_cross_entropy_with_logits(logits, soft_target, pos_weight=pos_weight_t)


def dice_loss(logits: torch.Tensor, hard_target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    probs = probs.flatten(1)
    target = hard_target.flatten(1)
    intersection = (probs * target).sum(dim=1)
    union = probs.sum(dim=1) + target.sum(dim=1)
    dice = (2 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def localization_loss(logits: torch.Tensor, soft_target: torch.Tensor, hard_target: torch.Tensor,
                       lambda_bce: float = 0.5, lambda_dice: float = 0.5) -> torch.Tensor:
    return (lambda_bce * weighted_bce_loss(logits, soft_target)
            + lambda_dice * dice_loss(logits, hard_target))


def detection_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(logits, target)


def total_loss(det_logits, loc_logits, y_det, soft_mask, hard_mask, alpha: float = 0.3,
               lambda_bce: float = 0.5, lambda_dice: float = 0.5):
    l_det = detection_loss(det_logits, y_det)
    l_loc = localization_loss(loc_logits, soft_mask, hard_mask, lambda_bce, lambda_dice)
    return l_det + alpha * l_loc, {"det": l_det.item(), "loc": l_loc.item()}


# quick check
_det_logits = torch.randn(4)
_loc_logits = torch.randn(4, 1, 64, 64)
_y_det = torch.randint(0, 2, (4,)).float()
_hard_mask = (torch.rand(4, 1, 64, 64) < 0.05).float()  # sparse, realistic
_soft_mask = _hard_mask.clone()
_loss, _parts = total_loss(_det_logits, _loc_logits, _y_det, _soft_mask, _hard_mask)
print("total loss:", _loss.item(), _parts)


total loss: 1.106431007385254 {'det': 0.7368595004081726, 'loc': 1.2319047451019287}


## 11. SRNet-style baseline (for your comparison table)

In [16]:
"""
A simplified SRNet-style baseline (detection-only), for the comparison table
your paper needs (Section 7.1 of the spec: "you must compare against SRNet or
a close variant"). This is a faithful-in-spirit, parameter-light version of
SRNet's structure (Boroumand et al., "Deep Residual Network for Steganalysis
of Digital Images," TIFS 2019) -- not a line-for-line reproduction of their
exact layer widths, which is fine for a baseline as long as you report its
param count honestly (see net.py's pattern of printing param counts).

Structure: unpooled high-pass-residual conv layers (types 1) -> residual
blocks without pooling (type 2) -> residual blocks with pooling (type 3) ->
final conv + global pooling -> FC classifier. No high-pass filter bank at the
front here on purpose -- SRNet's whole point is that it learns its own
low-level filters from scratch, unlike Xu-Net/Zhu-Net which hand-initialize
with SRM kernels. That's a real architectural difference worth keeping for a
fair, literature-accurate baseline.
"""
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    """Type 1: conv-BN-ReLU, no residual connection."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class ResBlockNoPool(nn.Module):
    """Type 2: two conv-BN layers with a residual add, no spatial reduction."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + x)


class ResBlockPool(nn.Module):
    """Type 3: residual block with a strided conv shortcut for downsampling."""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, stride=2, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=2, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class SRNetBaseline(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        c = base_channels
        self.stem = nn.Sequential(
            ConvBlock(1, c),
            ConvBlock(c, c),
        )
        self.no_pool = nn.Sequential(
            ResBlockNoPool(c),
            ResBlockNoPool(c),
            ResBlockNoPool(c),
            ResBlockNoPool(c),
            ResBlockNoPool(c),
        )
        self.pool_stages = nn.Sequential(
            ResBlockPool(c, c * 2),
            ResBlockPool(c * 2, c * 4),
            ResBlockPool(c * 4, c * 8),
            ResBlockPool(c * 8, c * 8),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c * 8, 1)

    def forward(self, x):
        f = self.stem(x)
        f = self.no_pool(f)
        f = self.pool_stages(f)
        z = self.gap(f).flatten(1)
        return self.fc(z).squeeze(-1)  # detection-only, matches DetectionHead's logit convention


# quick check
_srnet = SRNetBaseline()
_n = sum(p.numel() for p in _srnet.parameters())
print(f"SRNetBaseline params: {_n:,} ({_n/1e6:.2f}M)")
print("output logits:", _srnet(torch.randn(2, 1, 256, 256)).shape)


SRNetBaseline params: 640,465 (0.64M)


output logits: torch.Size([2])


## 12. Training loop (3-stage schedule)

Stage 1: detection-only (`alpha=0`) -- learn to detect first.
Stage 2: add localization loss with a small `alpha`.
Stage 3: fine-tune with the full multi-task loss.

Set `EPOCHS`/`STAGE1_EPOCHS`/`STAGE2_EPOCHS` properly for real training (60/20/20
is a reasonable starting point on BOSSbase); the small values below are just
for this notebook's smoke test.

In [17]:
def evaluate_quick(model, loader, device, max_batches=None):
    model.eval()
    n, correct = 0, 0
    dice_sum, dice_n = 0.0, 0
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches and i >= max_batches:
                break
            x = batch["image"].to(device)
            y_det = batch["y_det"].to(device)
            hard_mask = batch["hard_mask"].to(device)

            det_logits, loc_logits = model(x)
            pred = (torch.sigmoid(det_logits) > 0.5).float()
            correct += (pred == y_det).sum().item()
            n += y_det.numel()

            stego_idx = (y_det > 0.5) & (hard_mask.flatten(1).sum(1) > 0)
            if stego_idx.any():
                probs = torch.sigmoid(loc_logits[stego_idx])
                target = hard_mask[stego_idx]
                inter = (probs * target).flatten(1).sum(1)
                union = probs.flatten(1).sum(1) + target.flatten(1).sum(1)
                dice = (2 * inter + 1e-6) / (union + 1e-6)
                dice_sum += dice.sum().item()
                dice_n += dice.numel()
    model.train()
    return {"detection_acc": correct / max(n, 1),
            "localization_dice": dice_sum / max(dice_n, 1)}

In [18]:
import time, json, os

# ---- training config -- EDIT THESE for real runs ----
TRAIN_COVER_DIR = COVER_DIR       # point at your real train split
VAL_COVER_DIR = COVER_DIR         # point at your real val split (must not overlap train!)
OUT_DIR = "runs/exp1"
IMAGE_SIZE = 256
PAYLOADS = [0.4]                  # e.g. [0.1, 0.2, 0.3, 0.4] for real training
BATCH_SIZE = 2
NUM_WORKERS = 0
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 2                        # e.g. 60 for real training
STAGE1_EPOCHS = 1                 # e.g. 20
STAGE2_EPOCHS = 1                 # e.g. 20
STAGE2_ALPHA = 0.1
STAGE3_ALPHA = 0.3
EVAL_EVERY = 1
# ------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

model = SFMambaSteganalysis().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,} ({n_params/1e6:.2f}M)")

train_loader = get_dataloader(TRAIN_COVER_DIR, payload_bpps=PAYLOADS, image_size=IMAGE_SIZE,
                               batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
val_loader = get_dataloader(VAL_COVER_DIR, payload_bpps=PAYLOADS, image_size=IMAGE_SIZE,
                             batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def alpha_for_epoch(epoch):
    if epoch < STAGE1_EPOCHS:
        return 0.0
    elif epoch < STAGE1_EPOCHS + STAGE2_EPOCHS:
        return STAGE2_ALPHA
    else:
        return STAGE3_ALPHA

best_val_acc = 0.0
log_path = os.path.join(OUT_DIR, "log.jsonl")

for epoch in range(EPOCHS):
    alpha = alpha_for_epoch(epoch)
    t0 = time.time()
    running = {"total": 0.0, "det": 0.0, "loc": 0.0, "n": 0}

    for batch in train_loader:
        x = batch["image"].to(DEVICE)
        y_det = batch["y_det"].to(DEVICE)
        hard_mask = batch["hard_mask"].to(DEVICE)
        soft_mask = batch["soft_mask"].to(DEVICE)

        det_logits, loc_logits = model(x)
        loss, parts = total_loss(det_logits, loc_logits, y_det, soft_mask, hard_mask, alpha=alpha)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        running["total"] += loss.item() * bs
        running["det"] += parts["det"] * bs
        running["loc"] += parts["loc"] * bs
        running["n"] += bs

    scheduler.step()
    train_metrics = {k: running[k] / running["n"] for k in ["total", "det", "loc"]}
    log_entry = {"epoch": epoch, "alpha": alpha, "time_s": round(time.time() - t0, 1),
                 **{f"train_{k}": v for k, v in train_metrics.items()}}

    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS - 1:
        val_metrics = evaluate_quick(model, val_loader, DEVICE)
        log_entry.update({f"val_{k}": v for k, v in val_metrics.items()})
        if val_metrics["detection_acc"] > best_val_acc:
            best_val_acc = val_metrics["detection_acc"]
            torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_model.pt"))

    print(json.dumps(log_entry))
    with open(log_path, "a") as f:
        f.write(json.dumps(log_entry) + "\n")

torch.save(model.state_dict(), os.path.join(OUT_DIR, "final_model.pt"))
print("Training complete. Best val detection acc:", best_val_acc)

Model params: 416,279 (0.42M)


{"epoch": 0, "alpha": 0.0, "time_s": 7.4, "train_total": 0.6971180041631063, "train_det": 0.6971180041631063, "train_loc": 1.0234822034835815, "val_detection_acc": 0.6666666666666666, "val_localization_dice": 0.11249560117721558}


{"epoch": 1, "alpha": 0.1, "time_s": 6.8, "train_total": 0.8319907188415527, "train_det": 0.7279107968012491, "train_loc": 1.0407992601394653, "val_detection_acc": 0.3333333333333333, "val_localization_dice": 0.11118986643850803}
Training complete. Best val detection acc: 0.6666666666666666


## 13. Full evaluation (metrics your paper needs)

Detection accuracy/balanced-accuracy/ROC-AUC/precision/recall/F1, localization
pixel-precision/recall/F1/IoU/Dice, efficiency (params, ms/image) -- overall
**and per payload level**. Report per-payload in your paper, not just the
average -- averaging hides how badly a model degrades at low payload
(0.1-0.2 bpp), which reviewers will specifically ask about.

In [ ]:
1, Dice, IoU
  - Per-payload breakdown (report a table row per bpp, not just an average)
  - Efficiency: param count, FLOPs (approx, via thop if installed), ms/image

Usage:
    python evaluate.py --checkpoint runs/exp1/best_model.pt \
                        --test_cover_dir /path/to/test_cover
"""
import time
import json

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, balanced_accuracy_score

# (SFMambaSteganalysis / get_dataloader already defined above in this notebook)


def compute_localization_metrics(probs, hard_mask, threshold=0.5):
    pred = (probs > threshold)"""
Standalone evaluation: run a trained checkpoint against a test set and report
the metrics your paper actually needs (Section 7 of the spec):
  - Detection: accuracy, balanced accuracy, ROC-AUC, precision/recall/F1
  - Localization: pixel precision/recall/F.float()
    tp = (pred * hard_mask).sum().item()
    fp = (pred * (1 - hard_mask)).sum().item()
    fn = ((1 - pred) * hard_mask).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    return {"pixel_precision": precision, "pixel_recall": recall,
            "pixel_f1": f1, "iou": iou, "dice": dice}


@torch.no_grad()
def run_eval(model, loader, device, payload_filter=None):
    model.eval()
    y_true, y_score = [], []
    loc_metrics_accum = []
    inference_times = []

    for batch in loader:
        x = batch["image"].to(device)
        y_det = batch["y_det"]
        hard_mask = batch["hard_mask"].to(device)

        t0 = time.time()
        det_logits, loc_logits = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        inference_times.append((time.time() - t0) / x.size(0))

        y_true.extend(y_det.numpy().tolist())
        y_score.extend(torch.sigmoid(det_logits).cpu().numpy().tolist())

        stego_idx = (y_det > 0.5) & (hard_mask.flatten(1).sum(1).cpu() > 0)
        if stego_idx.any():
            probs = torch.sigmoid(loc_logits[stego_idx])
            target = hard_mask[stego_idx]
            loc_metrics_accum.append(compute_localization_metrics(probs, target))

    y_true = np.array(y_true)
    y_score = np.array(y_score)
    y_pred = (y_score > 0.5).astype(float)

    det_metrics = {
        "accuracy": float((y_pred == y_true).mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_score)) if len(set(y_true.tolist())) > 1 else float("nan"),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

    if loc_metrics_accum:
        loc_metrics = {k: float(np.mean([m[k] for m in loc_metrics_accum]))
                       for k in loc_metrics_accum[0]}
    else:
        loc_metrics = {}

    efficiency = {
        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
        "ms_per_image": float(np.mean(inference_times) * 1000),
    }

    return {"detection": det_metrics, "localization": loc_metrics, "efficiency": efficiency}


# call run_eval_all(...) in the cell below instead of main()/argparse


In [20]:
def run_eval_all(checkpoint_path, test_cover_dir, image_size=256, batch_size=16,
                  payloads=(0.1, 0.2, 0.3, 0.4), out_json=None):
    model = SFMambaSteganalysis().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))

    results = {}
    loader = get_dataloader(test_cover_dir, payload_bpps=list(payloads), image_size=image_size,
                             batch_size=batch_size, shuffle=False, num_workers=0)
    results["overall"] = run_eval(model, loader, DEVICE)

    for bpp in payloads:
        loader = get_dataloader(test_cover_dir, payload_bpps=[bpp], image_size=image_size,
                                 batch_size=batch_size, shuffle=False, num_workers=0)
        results[f"bpp_{bpp}"] = run_eval(model, loader, DEVICE)

    print(json.dumps(results, indent=2))
    if out_json:
        with open(out_json, "w") as f:
            json.dump(results, f, indent=2)
    return results

# run against the checkpoint we just trained above
_ = run_eval_all(
    checkpoint_path=os.path.join(OUT_DIR, "best_model.pt"),
    test_cover_dir=COVER_DIR,          # <-- point at your real held-out test split
    batch_size=2,
    payloads=(0.4,),
    out_json=os.path.join(OUT_DIR, "test_results.json"),
)

{
  "overall": {
    "detection": {
      "accuracy": 0.5,
      "balanced_accuracy": 0.5,
      "roc_auc": 0.0,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0
    },
    "localization": {
      "pixel_precision": 0.06266021728514906,
      "pixel_recall": 0.9999999999981781,
      "pixel_f1": 0.11793049909121436,
      "iou": 0.06266021728514906,
      "dice": 0.11793050020099069
    },
    "efficiency": {
      "params_M": 0.416279,
      "ms_per_image": 174.52426751454672
    }
  },
  "bpp_0.4": {
    "detection": {
      "accuracy": 0.6666666666666666,
      "balanced_accuracy": 0.5,
      "roc_auc": 0.125,
      "precision": 0.0,
      "recall": 0.0,
      "f1": 0.0
    },
    "localization": {
      "pixel_precision": 0.06377410888671388,
      "pixel_recall": 0.9999999999988036,
      "pixel_f1": 0.11990159894743764,
      "iou": 0.06377410888671388,
      "dice": 0.11990160007458026
    },
    "efficiency": {
      "params_M": 0.416279,
      "ms_per_image": 175.0

## 14. One-time train/val/test split helper

Run this once on your real BOSSbase folder before training for real (not
needed for the synthetic smoke test above, since it already ran on the same
6 images for train/val -- don't do that for real experiments, it's leakage).

In [21]:
"""
Split a directory of cover images into train/val/test by filename, with no
overlap. Run this ONCE before training -- train.py and evaluate.py assume
you're pointing them at already-separated directories, on purpose, so a
train/test leakage bug can't hide inside the training script.

Usage:
    python data/split_dataset.py --cover_dir /path/to/BOSSbase_1.01 \
                                  --out_dir /path/to/split \
                                  --train_frac 0.7 --val_frac 0.15
"""
import argparse
import os
import random
import shutil
import glob


def split_dataset(cover_dir, out_dir, train_frac=0.7, val_frac=0.15, seed=42,
                   extensions=(".pgm", ".png", ".pnm"), copy=False):
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(cover_dir, f"*{ext}")))
    paths = sorted(paths)
    if not paths:
        raise FileNotFoundError(f"No images found in {cover_dir}")

    rng = random.Random(seed)
    rng.shuffle(paths)

    n = len(paths)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    splits = {"train": paths[:n_train], "val": paths[n_train:n_train + n_val],
              "test": paths[n_train + n_val:]}

    for split_name, split_paths in splits.items():
        split_dir = os.path.join(out_dir, split_name)
        os.makedirs(split_dir, exist_ok=True)
        for src in split_paths:
            dst = os.path.join(split_dir, os.path.basename(src))
            if copy:
                shutil.copy2(src, dst)
            else:
                if os.path.lexists(dst):
                    os.remove(dst)
                os.symlink(os.path.abspath(src), dst)
        print(f"{split_name}: {len(split_paths)} images -> {split_dir}")
    print("Done. No filename appears in more than one split.")

# Example usage (uncomment and point at your real BOSSbase folder):
# split_dataset(cover_dir="/path/to/BOSSbase_1.01", out_dir="/path/to/split",
#               train_frac=0.7, val_frac=0.15)


## What's verified vs. not

**Verified on this machine** (no GPU / BOSSbase available here):
- Model forward + backward pass: correct output shapes, gradients flow into
  every trainable parameter, high-pass stem correctly stays frozen.
- HILL cost + simulator: confirmed content-adaptive behavior -- on a synthetic
  half-textured/half-smooth test image, 98-100% of simulated changes at every
  tested payload landed in the textured half, not the smooth half.
- Full pipeline: ran an actual training loop end-to-end (data generation,
  detection-only stage, localization-added stage, checkpointing, logging) and
  then ran the full evaluation script against the resulting checkpoint,
  producing every metric without errors.

**NOT verified**: real detection accuracy, whether the Mamba branch helps vs.
hurts on real images, hyperparameter tuning, training speed/stability at full
BOSSbase scale. All of that needs your GPU and real data.

## Before scaling to full BOSSbase training

1. **Swap the selective scan for the real CUDA kernel.** `_selective_scan_sequential`
   in the VSS block section is a correct but slow pure-PyTorch Python loop.
   Install `mamba-ssm` and swap in `mamba_ssm.ops.selective_scan_fn` for real
   training speed (10-50x reported speedups) once you have a CUDA GPU.
2. **Only HILL is implemented.** WOW/S-UNIWARD use directional filter banks
   and need separate implementation for your generalization-across-algorithms
   experiments -- budget real time for this.
3. **Hyperparameters here are reasonable starting points, not tuned values** --
   expect your ablation runs to also serve as a hyperparameter search.
4. **Point `TRAIN_COVER_DIR`/`VAL_COVER_DIR`/test dir at three non-overlapping
   splits** from `split_dataset()` above -- the smoke test above reuses the
   same 6 synthetic images for everything, which is leakage; fine for
   verifying the code runs, not fine for real results.